# Klasifikasi ordinal: label berurutan (rating 1–5, tingkat keparahan, kelompok usia)
**Kapan dipakai:** kelas punya urutan, salah satu tingkat lebih ringan daripada salah jauh, dan
biasanya dinilai QWK (quadratic weighted kappa) atau MAE. Enam cara di lipatan yang sama:

1. **datar**: klasifikasi biasa, argmax (urutan diabaikan).
2. **kumulatif argmax** (Frank & Hall): K−1 model biner "apakah > tingkat k?". Peluang tiap
   tingkat = selisih peluang berurutan.
3. **regresi + ambang**: Ridge menebak nomor tingkat, ambang pembulatan dicari di OOF.
4. **kumulatif + ambang**: nilai harapan Σ P(> k), lalu ambang.
5. **harapan datar + ambang**: Σ peluang × nomor tingkat dari model datar, lalu ambang.
6. **gabung + ambang**: rata-rata nilai harapan cara 3–5, lalu ambang.

Ambang dicari dengan pencarian koordinat (tiap ambang digeser bergantian) di seluruh OOF, jadi
skor OOF cara ber-ambang sedikit optimis. Pembanding jujur: cek kunci test di akhir.
**Data demo:** ulasan baju, `rating` 1–5.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi
`LEVEL_ORDER`: urutan tingkat kalau labelnya teks (mis. `["ringan", "sedang", "berat"]`).
`None` = urutan angka label.

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/ordinal_teks"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample panitia
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                # GANTI saat lomba
TARGET = "rating"                            # GANTI saat lomba
SUB_COL = "rating"                           # GANTI saat lomba: kolom sample
TEXT_COLS = ["title", "review_text"]         # GANTI saat lomba
LEVEL_ORDER = None                           # GANTI kalau label teks
MODEL_NAME = "tfidf_ordinal"
N_FOLDS = 5
SEED = 42


def metric(truth_idx, pred_idx):
    return cohen_kappa_score(truth_idx, pred_idx, weights="quadratic")   # GANTI: metrik lomba
    # return -np.abs(truth_idx - pred_idx).mean()                       # alt: -MAE
    # return f1_score(truth_idx, pred_idx, average="macro")             # alt: F1 makro


def make_classifier():
    return SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
    # return LogisticRegression(C=4.0, max_iter=2000)   # alt: peluang lebih halus, lebih lambat


print("uji metrik:", metric(np.array([0, 1, 2, 3]), np.array([0, 1, 3, 3])))

## Baca data

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
LEVELS = np.sort(train[TARGET].unique())
if LEVEL_ORDER is not None:
    LEVELS = np.array(LEVEL_ORDER, dtype=object)
N_LEVELS = len(LEVELS)
level_index = pd.Series(np.arange(N_LEVELS), index=LEVELS)
y_idx = train[TARGET].map(level_index).to_numpy()
print("tingkat:", list(LEVELS), "| jumlah:", np.bincount(y_idx))

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = LEVELS[0]
demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Lipatan + TF-IDF

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train, y_idx))
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
all_text = pd.concat([full_train, full_test], ignore_index=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=150000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=150000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
print("TF-IDF:", matrix_train.shape)

## Model dasar: datar, Ridge, dan K−1 model biner kumulatif

In [ ]:
def oof_proba(labels, width):
    oof = np.zeros((len(train), width))
    test_pred = np.zeros((len(test), width))
    for idx_fit, idx_val in folds:
        fitted = make_classifier().fit(matrix_train[idx_fit], labels[idx_fit])
        oof[np.ix_(idx_val, fitted.classes_)] = fitted.predict_proba(matrix_train[idx_val])
        test_pred[:, fitted.classes_] += fitted.predict_proba(matrix_test) / N_FOLDS
    return oof, test_pred


START = time.time()
flat_oof, flat_test = oof_proba(y_idx, N_LEVELS)
print("datar selesai | detik", round(time.time() - START))

In [ ]:
def oof_value(labels):
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    for idx_fit, idx_val in folds:
        fitted = Ridge(alpha=2.0).fit(matrix_train[idx_fit], labels[idx_fit])
        oof[idx_val] = fitted.predict(matrix_train[idx_val])
        test_pred += fitted.predict(matrix_test) / N_FOLDS
    return oof, test_pred


ridge_oof, ridge_test = oof_value(y_idx.astype(float))
above_oof = np.zeros((len(train), N_LEVELS - 1))
above_test = np.zeros((len(test), N_LEVELS - 1))
for level in range(N_LEVELS - 1):
    binary_oof, binary_test = oof_proba((y_idx > level).astype(int), 2)
    above_oof[:, level] = binary_oof[:, 1]
    above_test[:, level] = binary_test[:, 1]
SECONDS = round(time.time() - START)
print("model dasar selesai | detik", SECONDS)

## Kumulatif → peluang tiap tingkat
P(> k) dipaksa turun (`minimum.accumulate`), P(= k) = P(> k−1) − P(> k).
Nilai harapan nomor tingkat = Σ P(> k).

In [ ]:
def cumulative_to_proba(above):
    above = np.minimum.accumulate(above, axis=1)
    upper = np.hstack([np.ones((len(above), 1)), above])
    lower = np.hstack([above, np.zeros((len(above), 1))])
    proba = np.clip(upper - lower, 1e-6, None)
    return proba / proba.sum(axis=1, keepdims=True)


cum_oof = cumulative_to_proba(above_oof)
cum_test = cumulative_to_proba(above_test)
steps = np.arange(N_LEVELS)
VALUES = {"regresi + ambang": (ridge_oof, ridge_test),
          "kumulatif + ambang": (cum_oof @ steps, cum_test @ steps),
          "harapan datar + ambang": (flat_oof @ steps, flat_test @ steps)}
gabung_oof = (ridge_oof + cum_oof @ steps + flat_oof @ steps) / 3
gabung_test = (ridge_test + cum_test @ steps + flat_test @ steps) / 3
VALUES["gabung + ambang"] = (gabung_oof, gabung_test)
print("jumlah peluang per baris:", cum_oof.sum(axis=1)[:3])

## Ambang pembulatan (pencarian koordinat)

In [ ]:
def apply_cuts(values, cuts):
    return np.searchsorted(np.sort(cuts), values)


def fit_cuts(values, truth_idx):
    cuts = np.arange(1, N_LEVELS) - 0.5
    grid = np.unique(np.quantile(values, np.linspace(0.005, 0.995, 120)))
    for _ in range(3):
        for position in range(len(cuts)):
            best_score = metric(truth_idx, apply_cuts(values, cuts))
            for candidate in grid:
                trial = cuts.copy()
                trial[position] = candidate
                score = metric(truth_idx, apply_cuts(values, trial))
                if score > best_score:
                    best_score = score
                    cuts = trial
    return np.sort(cuts)


PRED = {"datar": (flat_oof.argmax(axis=1), flat_test.argmax(axis=1)),
        "kumulatif argmax": (cum_oof.argmax(axis=1), cum_test.argmax(axis=1))}
CUTS = {}
for name in VALUES:
    CUTS[name] = fit_cuts(VALUES[name][0], y_idx)
    PRED[name] = (apply_cuts(VALUES[name][0], CUTS[name]), apply_cuts(VALUES[name][1], CUTS[name]))
    print(name, "| ambang:", np.round(CUTS[name], 3))

## Banding cara (bahan slide)
"salah ≥ 2 tingkat" = porsi baris yang meleset jauh: inilah yang dihukum berat oleh QWK.

In [ ]:
def describe(truth_idx, pred_idx):
    gap = np.abs(truth_idx - pred_idx)
    return {"metrik": round(metric(truth_idx, pred_idx), 4), "MAE tingkat": round(gap.mean(), 4),
            "akurasi": round((gap == 0).mean(), 4),
            "F1 makro": round(f1_score(truth_idx, pred_idx, average="macro"), 4),
            "salah ≥ 2 tingkat": round((gap >= 2).mean(), 4)}


rows = []
for name in PRED:
    rows.append(describe(y_idx, PRED[name][0]))
table = pd.DataFrame(rows, index=list(PRED))
table.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv")
BEST = table["metrik"].idxmax()      # alt: tulis manual, mis. "kumulatif + ambang"
print(table)
print("dipilih:", BEST)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(table.index, table["metrik"], color="#0072B2")
axes[0].set_ylim(table["metrik"].min() - 0.03, table["metrik"].max() + 0.01)
axes[0].set_ylabel("metrik OOF")
axes[0].tick_params(axis="x", rotation=25)
confusion = pd.crosstab(LEVELS[y_idx], LEVELS[PRED[BEST][0]], normalize="index")
axes[1].imshow(confusion.to_numpy(), cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks(range(confusion.shape[1]), confusion.columns)
axes[1].set_yticks(range(confusion.shape[0]), confusion.index)
for row in range(confusion.shape[0]):
    for col in range(confusion.shape[1]):
        axes[1].text(col, row, f"{confusion.iat[row, col]:.2f}", ha="center", va="center")
axes[1].set_xlabel(f"tebakan ({BEST})")
axes[1].set_ylabel("asli")
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/banding_dan_bingung.png", dpi=200, bbox_inches="tight")
plt.show()

## Simpan OOF + catat percobaan
Disimpan peluang kumulatif (n × K, kolom = `np.unique(train[TARGET])`) supaya bisa masuk
`ensemble_oof`. Ambang tetap dicari ulang setelah ensemble.

In [ ]:
unique_levels = np.unique(train[TARGET])
columns = level_index.loc[unique_levels].to_numpy()
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", cum_oof[:, columns].astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", cum_test[:, columns].astype("float32"))
log_path = f"{OOF_FOLDER}/percobaan.csv"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "ordinal_teks",
                         "model": MODEL_NAME, "metrik": "qwk", "skor_cv": table.loc[BEST, "metrik"],
                         "detik": SECONDS, "catatan": BEST}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("disimpan:", f"oof_{MODEL_NAME}.npy", cum_oof.shape)

## Submission + cek

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(LEVELS[PRED[BEST][1]], index=test[ID_COL].values)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
shares = written[SUB_COL].value_counts(normalize=True)
print(shares.round(3).to_dict())
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = test[ID_COL].map(key_by_id)
truth_idx = truth.map(level_index).to_numpy()
rows = []
for name in PRED:
    rows.append(describe(truth_idx, PRED[name][1]))
print(pd.DataFrame(rows, index=list(PRED)), "\nOOF terpilih:", BEST)